In [ ]:
!pip install optuna shap xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 6.9 MB/s eta 0:00:00


In [ ]:
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("optuna").setLevel(logging.WARNING)
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
import shap
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter


# CONFIGURATION
CSV_PATH      = "Dengue_Climate_Bangladesh.csv"
OUTPUT_PNG    = "xgboost_dengue_2025_forecast.png"
OPTUNA_TRIALS = 150
CV_FOLDS      = 5
RANDOM_STATE  = 42

P = dict(
    BG="#0d1117", PANEL="#161b22", GRID="#21262d", BORDER="#30363d",
    WHITE="#e6edf3", MUTED="#8b949e", BLUE="#58a6ff", ORANGE="#f78166",
    GREEN="#3fb950", YELLOW="#d29922", TEAL="#39c5cf", PURPLE="#bc8cff",
    RED="#ff6b6b", PINK="#f0a5c0",
)


# SECTION 1 ─ FEATURE ENGINEERING
def engineer_features(path: str) -> pd.DataFrame:
    """
    Load the raw panel and construct the complete feature matrix.

    Features
    --------
    Climate lags 1–3   : Mosquito breeding cycle delay proxies.
    Rolling climate    : 3-month smoothed signal for trend persistence.
    Cumulative rain    : Standing-water larval habitat proxy.
    Heat × humidity    : Transmission-environment interaction.
    Circular month     : sin/cos preserves Jan–Dec periodicity.
    Log-dengue lags    : Compressed autoregressive epidemic memory.
    12-month log-mean  : Stable trend baseline across regimes.
    5-yr climatology   : Endemic monthly baseline per calendar month.
    Anomaly index      : Departure of lag-12 log-dengue from climatology.
    """
    df = pd.read_csv(path)
    df["DATE"] = pd.to_datetime(
        df["YEAR"].astype(str) + "-" + df["MONTH"].astype(str) + "-01"
    )
    df = df.sort_values("DATE").reset_index(drop=True).set_index("DATE")
    df.index.freq = "MS"

    df["LOG_DENGUE"] = np.log1p(df["DENGUE"])
    df["TEMP_RANGE"] = df["MAX"] - df["MIN"]
    df["TEMP_MEAN"]  = (df["MAX"] + df["MIN"]) / 2
    df["HEAT_HUMID"] = df["TEMP_MEAN"] * df["HUMIDITY"]

    for col in ["RAINFALL", "HUMIDITY", "MIN", "MAX", "TEMP_RANGE"]:
        for lag in [1, 2, 3]:
            df[f"{col}_lag{lag}"] = df[col].shift(lag)

    df["RAINFALL_roll3"]  = df["RAINFALL"].rolling(3).mean()
    df["HUMIDITY_roll3"]  = df["HUMIDITY"].rolling(3).mean()
    df["RAINFALL_cum3"]   = df["RAINFALL"].rolling(3).sum()
    df["month_sin"]       = np.sin(2 * np.pi * df.index.month / 12)
    df["month_cos"]       = np.cos(2 * np.pi * df.index.month / 12)

    for lag in [1, 2, 3]:
        df[f"LOG_DENGUE_lag{lag}"] = df["LOG_DENGUE"].shift(lag)
    df["LOG_DENGUE_roll12"] = df["LOG_DENGUE"].rolling(12).mean()

    # 5-year rolling calendar-month climatology (no leakage)
    clim = []
    for idx in df.index:
        past = df.loc[
            (df.index.month == idx.month) &
            (df.index < idx) &
            (df.index >= idx - pd.DateOffset(years=5)),
            "LOG_DENGUE",
        ]
        clim.append(past.mean() if len(past) > 0 else np.nan)
    df["LOG_DENGUE_clim5y"] = clim

    lag12 = df["LOG_DENGUE"].shift(12)
    df["DENGUE_anomaly"] = lag12 - df["LOG_DENGUE_clim5y"]

    df.drop(columns=["LOG_DENGUE"], inplace=True)
    df.dropna(inplace=True)
    return df


FEATURE_COLS = [
    "MIN", "MAX", "HUMIDITY", "RAINFALL", "TEMP_RANGE", "TEMP_MEAN",
    "RAINFALL_lag1", "RAINFALL_lag2", "RAINFALL_lag3",
    "HUMIDITY_lag1", "HUMIDITY_lag2", "HUMIDITY_lag3",
    "MIN_lag1", "MIN_lag2", "MIN_lag3",
    "MAX_lag1", "MAX_lag2",
    "TEMP_RANGE_lag1", "TEMP_RANGE_lag2",
    "RAINFALL_roll3", "HUMIDITY_roll3", "RAINFALL_cum3",
    "HEAT_HUMID", "month_sin", "month_cos",
    "LOG_DENGUE_lag1", "LOG_DENGUE_lag2", "LOG_DENGUE_lag3",
    "LOG_DENGUE_roll12", "LOG_DENGUE_clim5y", "DENGUE_anomaly",
]

FEATURE_LABELS = {
    "MIN": "Min Temp", "MAX": "Max Temp", "HUMIDITY": "Humidity",
    "RAINFALL": "Rainfall", "TEMP_RANGE": "Temp Range", "TEMP_MEAN": "Mean Temp",
    "RAINFALL_lag1": "Rainfall lag-1", "RAINFALL_lag2": "Rainfall lag-2",
    "RAINFALL_lag3": "Rainfall lag-3",
    "HUMIDITY_lag1": "Humidity lag-1", "HUMIDITY_lag2": "Humidity lag-2",
    "HUMIDITY_lag3": "Humidity lag-3",
    "MIN_lag1": "Min Temp lag-1", "MIN_lag2": "Min Temp lag-2",
    "MIN_lag3": "Min Temp lag-3",
    "MAX_lag1": "Max Temp lag-1", "MAX_lag2": "Max Temp lag-2",
    "TEMP_RANGE_lag1": "Temp Range lag-1", "TEMP_RANGE_lag2": "Temp Range lag-2",
    "RAINFALL_roll3": "Rainfall (3mo avg)", "HUMIDITY_roll3": "Humidity (3mo avg)",
    "RAINFALL_cum3": "Rainfall (3mo cum)", "HEAT_HUMID": "Heat × Humidity",
    "month_sin": "Seasonality (sin)", "month_cos": "Seasonality (cos)",
    "LOG_DENGUE_lag1": "Log Dengue lag-1", "LOG_DENGUE_lag2": "Log Dengue lag-2",
    "LOG_DENGUE_lag3": "Log Dengue lag-3",
    "LOG_DENGUE_roll12": "Log Dengue (12mo avg)",
    "LOG_DENGUE_clim5y": "Log Dengue (5yr clim.)",
    "DENGUE_anomaly": "Epidemic Anomaly Index",
}


# SECTION 2 ─ OPTUNA HPO
def optimise(X_tr: pd.DataFrame, y_tr_log: np.ndarray) -> dict:
    """
    Bayesian HPO minimising CV-MAE evaluated in original count scale.
    TimeSeriesSplit ensures strict temporal ordering across all folds.
    """
    tscv = TimeSeriesSplit(n_splits=CV_FOLDS)

    def objective(trial):
        p = {
            "n_estimators":      trial.suggest_int("n_estimators", 80, 600),
            "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.15, log=True),
            "max_depth":         trial.suggest_int("max_depth", 2, 5),
            "min_child_weight":  trial.suggest_int("min_child_weight", 3, 30),
            "subsample":         trial.suggest_float("subsample", 0.50, 0.90),
            "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.40, 0.90),
            "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.40, 0.90),
            "gamma":             trial.suggest_float("gamma", 0.5, 8.0),
            "reg_alpha":         trial.suggest_float("reg_alpha", 0.01, 20.0, log=True),
            "reg_lambda":        trial.suggest_float("reg_lambda", 0.01, 20.0, log=True),
            "objective": "reg:squarederror",
            "random_state": RANDOM_STATE, "n_jobs": -1,
        }
        maes = []
        for ti, vi in tscv.split(X_tr):
            m = xgb.XGBRegressor(**p)
            m.fit(X_tr.iloc[ti], y_tr_log[ti],
                  eval_set=[(X_tr.iloc[vi], y_tr_log[vi])], verbose=False)
            pred = np.clip(m.predict(X_tr.iloc[vi]), 0, None)
            maes.append(mean_absolute_error(np.expm1(y_tr_log[vi]), np.expm1(pred)))
        return np.mean(maes)

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )
    study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=False)
    best = study.best_params
    best.update({"objective": "reg:squarederror",
                 "random_state": RANDOM_STATE, "n_jobs": -1})
    return best


# SECTION 3 ─ METRICS
def metrics(y_true, y_pred):
    return dict(
        MAE       = mean_absolute_error(y_true, y_pred),
        RMSE      = np.sqrt(mean_squared_error(y_true, y_pred)),
        R2        = r2_score(y_true, y_pred),
        MAPE      = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1)) * 100),
        SMAPE     = np.mean(2 * np.abs(y_true - y_pred) /
                            (np.abs(y_true) + np.abs(y_pred) + 1e-9) * 100),
        Pearson_r = np.corrcoef(y_true, y_pred)[0, 1],
        Bias      = np.mean(y_pred - y_true),
    )


def print_report(tr_m, te_m, best_params, pred_dates, y_pred, y_test, n_tr):
    sep = "=" * 70
    print(f"\n{sep}")
    print("  XGBoost Dengue Forecasting  —  2025 Jan–Oct Prospective Evaluation")
    print(sep)
    print(f"  Target   : log₁p(DENGUE) → back-transformed with expm₁()")
    print(f"  Train    : 2008 → 2024  (n = {n_tr})")
    print(f"  Test     : Jan 2025 → Oct 2025  (n = {len(pred_dates)})")
    print()
    fmt  = {"MAE":",.1f","RMSE":",.1f","R2":".4f",
            "MAPE":".2f","SMAPE":".2f","Pearson_r":".4f","Bias":",.1f"}
    unit = {"MAPE":"%","SMAPE":"%"}
    print(f"  {'Metric':<14} {'In-Sample (Train)':>20}  {'Out-of-Sample (Test)':>22}")
    print(f"  {'-'*60}")
    for k, f in fmt.items():
        u = unit.get(k, "")
        print(f"  {k:<14} {format(tr_m[k], f)+u:>20}  {format(te_m[k], f)+u:>22}")
    print()
    print(f"  {'Month':<12} {'Predicted':>12} {'Observed':>12} {'Abs Err':>10} {'Err %':>8}")
    print(f"  {'-'*58}")
    for dt, p, o in zip(pred_dates, y_pred, y_test):
        pct  = abs(p - o) / max(o, 1) * 100
        flag = " ✓" if pct <= 30 else (" ≈" if pct <= 60 else " ✗")
        print(f"  {dt.strftime('%b %Y'):<12} {int(p):>12,} {int(o):>12,} "
              f"{int(abs(p-o)):>10,} {pct:>7.1f}%{flag}")
    total_pred = int(y_pred.sum())
    total_obs  = int(y_test.sum())
    print(f"\n  {'TOTAL (Jan–Oct)':<12} {total_pred:>12,} {total_obs:>12,} "
          f"{abs(total_pred-total_obs):>10,} "
          f"{abs(total_pred-total_obs)/total_obs*100:>7.1f}%")
    print(sep + "\n")


# SECTION 4 ─ VISUALISATION
def style_ax(ax, title, xlabel=None, ylabel=None, ts=10):
    ax.set_facecolor(P["PANEL"])
    ax.set_title(title, color=P["WHITE"], fontsize=ts, fontweight="bold", pad=9)
    ax.tick_params(colors=P["MUTED"], labelsize=8.5)
    ax.spines[["top","right"]].set_visible(False)
    ax.spines[["bottom","left"]].set_color(P["BORDER"])
    ax.grid(True, color=P["GRID"], ls="--", lw=0.55, alpha=0.9)
    if xlabel: ax.set_xlabel(xlabel, color=P["MUTED"], fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, color=P["MUTED"], fontsize=9)

def cfmt(x, _): return f"{int(x):,}"


def build_figure(df, train, test, y_train_raw, y_test,
                 y_fitted, y_pred, model,
                 tr_m, te_m, best_params, shap_values):

    pred_dates = test.index
    n_train    = len(train)

    fig = plt.figure(figsize=(22, 18), dpi=150)
    fig.patch.set_facecolor(P["BG"])
    gs = gridspec.GridSpec(3, 3, figure=fig,
                           hspace=0.44, wspace=0.30,
                           left=0.06, right=0.79, top=0.95, bottom=0.05)
    ax1 = fig.add_subplot(gs[0, :])
    ax2 = fig.add_subplot(gs[1, :2])
    ax3 = fig.add_subplot(gs[1, 2])
    ax4 = fig.add_subplot(gs[2, :2])
    ax5 = fig.add_subplot(gs[2, 2])

    #Panel 1 : Full Timeline
    style_ax(ax1,
        "Full Epidemiological Timeline — Bangladesh Dengue Incidence & XGBoost 2025 Forecast",
        xlabel="Year", ylabel="Monthly Dengue Incidence", ts=11)

    hist = df["DENGUE"].loc[:"2024-12-01"]
    ax1.fill_between(hist.index, hist.values, alpha=0.10, color=P["MUTED"])
    ax1.plot(hist.index, hist.values, color=P["MUTED"],
             lw=1.1, alpha=0.50, label="Historical Record (2008–2024)")
    ax1.plot(train.index, y_fitted, color=P["TEAL"],
             lw=0.9, ls="-.", alpha=0.60, label="In-Sample Fit")
    ax1.plot(pred_dates, y_test, color=P["BLUE"],
             lw=2.5, marker="o", ms=6, label="Observed 2025 (Jan–Oct)")
    ax1.plot(pred_dates, y_pred, color=P["ORANGE"],
             lw=2.5, ls="--", marker="x", ms=7.5,
             label="XGBoost Forecast 2025")
    ax1.axvline(pd.Timestamp("2025-01-01"), color=P["YELLOW"],
                lw=1.3, ls=":", alpha=0.9, label="Train / Test Boundary")

    # Annotate 2023 peak
    peak_idx = df["DENGUE"].idxmax()
    ax1.annotate(f"2023 Peak\n{df['DENGUE'].max():,}",
                 xy=(peak_idx, df["DENGUE"].max()),
                 xytext=(pd.Timestamp("2021-06-01"), df["DENGUE"].max() * 0.85),
                 color=P["RED"], fontsize=8,
                 arrowprops=dict(arrowstyle="->", color=P["RED"], lw=1.2))

    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax1.yaxis.set_major_formatter(FuncFormatter(cfmt))
    ax1.legend(fontsize=8.5, facecolor=P["PANEL"], labelcolor=P["WHITE"],
               framealpha=0.92, edgecolor=P["BORDER"],
               loc="upper left", ncol=3)

    #Panel 2 : 2025 Zoom with % error labels
    style_ax(ax2,
        "Prospective Validation (Jan–Oct 2025) — Monthly Error Decomposition",
        xlabel="Month (2025)", ylabel="Dengue Incidence")

    ax2.fill_between(pred_dates, y_test, y_pred, alpha=0.10, color=P["ORANGE"])
    ax2.plot(pred_dates, y_test, color=P["BLUE"],
             lw=2.6, marker="o", ms=7, zorder=5, label="Observed (DGHS 2025)")
    ax2.plot(pred_dates, y_pred, color=P["ORANGE"],
             lw=2.6, ls="--", marker="x", ms=8, zorder=5,
             label="XGBoost Forecast")

    y_ceil = max(y_test.max(), y_pred.max()) * 1.15
    for dt, p, o in zip(pred_dates, y_pred, y_test):
        pct = abs(p - o) / max(o, 1) * 100
        clr = P["GREEN"] if pct <= 30 else (P["YELLOW"] if pct <= 60 else P["RED"])
        ax2.text(dt, y_ceil * 0.97, f"{pct:.0f}%",
                 ha="center", va="bottom", fontsize=8.5,
                 color=clr, fontweight="bold")

    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    ax2.xaxis.set_major_locator(mdates.MonthLocator())
    ax2.yaxis.set_major_formatter(FuncFormatter(cfmt))
    ax2.set_ylim(0, y_ceil * 1.10)
    ax2.legend(fontsize=9, facecolor=P["PANEL"], labelcolor=P["WHITE"],
               framealpha=0.92, edgecolor=P["BORDER"], loc="upper left")

    # Running cumulative total annotation
    cumobs  = np.cumsum(y_test)
    cumpred = np.cumsum(y_pred)
    ax2b = ax2.twinx()
    ax2b.set_facecolor("none")
    ax2b.plot(pred_dates, cumobs,  color=P["BLUE"],   lw=1.2, ls=":", alpha=0.6)
    ax2b.plot(pred_dates, cumpred, color=P["ORANGE"], lw=1.2, ls=":", alpha=0.6)
    ax2b.set_ylabel("Cumulative Cases", color=P["MUTED"], fontsize=8)
    ax2b.tick_params(colors=P["MUTED"], labelsize=7.5)
    ax2b.spines[["top","right","bottom","left"]].set_color(P["BORDER"])
    ax2b.yaxis.set_major_formatter(FuncFormatter(cfmt))

    #Panel 3 : Scatter
    style_ax(ax3, "Predicted vs Observed",
             xlabel="Observed Counts", ylabel="Predicted Counts")
    ax3.scatter(y_train_raw, y_fitted, color=P["TEAL"],
                alpha=0.20, s=20, label="Train (2008–2024)")
    ax3.scatter(y_test, y_pred, color=P["ORANGE"],
                alpha=0.95, s=80, edgecolors=P["WHITE"],
                linewidths=0.7, zorder=5, label="Test (2025 Jan–Oct)")
    for dt, p, o in zip(pred_dates, y_pred, y_test):
        ax3.annotate(dt.strftime("%b"), (o, p), fontsize=7.5,
                     color=P["WHITE"], xytext=(4, 4),
                     textcoords="offset points")
    lim = max(y_train_raw.max(), y_test.max(),
              y_fitted.max(), y_pred.max()) * 1.06
    ax3.plot([0, lim], [0, lim], color=P["WHITE"], lw=1.1,
             ls="--", alpha=0.45, label="Perfect Fit (y = x̂)")
    ax3.text(0.05, 0.90,
             f"R²  (test) = {te_m['R2']:.3f}\n"
             f"r    (test) = {te_m['Pearson_r']:.3f}\n"
             f"SMAPE      = {te_m['SMAPE']:.1f}%",
             transform=ax3.transAxes, fontsize=8.5, color=P["WHITE"],
             bbox=dict(boxstyle="round,pad=0.4",
                       facecolor=P["BORDER"], alpha=0.85))
    ax3.set_xlim(0, lim); ax3.set_ylim(0, lim)
    ax3.xaxis.set_major_formatter(FuncFormatter(cfmt))
    ax3.yaxis.set_major_formatter(FuncFormatter(cfmt))
    ax3.legend(fontsize=8.5, facecolor=P["PANEL"], labelcolor=P["WHITE"],
               framealpha=0.92, edgecolor=P["BORDER"])

    #Panel 4 : Monthly Bar Comparison
    style_ax(ax4,
        "Monthly Count Comparison — Observed vs XGBoost Forecast (Jan–Oct 2025)",
        xlabel="Month (2025)", ylabel="Dengue Incidence")
    idx = np.arange(len(test))
    bw  = 0.36
    months_lbl = [dt.strftime("%b") for dt in pred_dates]
    b_obs  = ax4.bar(idx - bw/2, y_test,  bw, color=P["BLUE"],
                     alpha=0.85, label="Observed (DGHS)", zorder=3)
    b_pred = ax4.bar(idx + bw/2, y_pred, bw, color=P["ORANGE"],
                     alpha=0.85, label="XGBoost Forecast", zorder=3)
    for bar, val in zip(b_obs, y_test):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                 f"{int(val):,}", ha="center", va="bottom",
                 fontsize=6.5, color=P["BLUE"])
    for bar, val in zip(b_pred, y_pred):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                 f"{int(val):,}", ha="center", va="bottom",
                 fontsize=6.5, color=P["ORANGE"])
    ax4.set_xticks(idx)
    ax4.set_xticklabels(months_lbl, fontsize=9)
    ax4.yaxis.set_major_formatter(FuncFormatter(cfmt))
    ax4.legend(fontsize=9, facecolor=P["PANEL"], labelcolor=P["WHITE"],
               framealpha=0.92, edgecolor=P["BORDER"], loc="upper left")

    # Cumulative bar totals annotation
    ax4.text(0.99, 0.97,
             f"Σ Observed : {int(y_test.sum()):,}\n"
             f"Σ Forecast : {int(y_pred.sum()):,}\n"
             f"Total Err  : {abs(int(y_pred.sum())-int(y_test.sum())):,} "
             f"({abs(y_pred.sum()-y_test.sum())/y_test.sum()*100:.1f}%)",
             transform=ax4.transAxes, ha="right", va="top",
             fontsize=8.5, color=P["WHITE"], fontfamily="monospace",
             bbox=dict(boxstyle="round,pad=0.4",
                       facecolor=P["BORDER"], alpha=0.85))

    #Panel 5 : SHAP Feature Importance─
    style_ax(ax5, "SHAP Mean |Value| — Top 15 Features",
             xlabel="Mean |SHAP Value|", ylabel="Feature")
    mean_shap = np.abs(shap_values).mean(axis=0)
    imp_df = pd.DataFrame({
        "label": [FEATURE_LABELS.get(c, c) for c in FEATURE_COLS],
        "shap":  mean_shap,
    }).sort_values("shap", ascending=True).tail(15)
    yp   = np.arange(len(imp_df))
    q75  = imp_df["shap"].quantile(0.75)
    cols = [P["PURPLE"] if v >= q75 else P["TEAL"] for v in imp_df["shap"]]
    ax5.barh(yp, imp_df["shap"], color=cols, alpha=0.85, height=0.65, zorder=3)
    ax5.set_yticks(yp)
    ax5.set_yticklabels(imp_df["label"], fontsize=8, color=P["MUTED"])

    #Sidebar Card
    card = [
        "╔══ XGBoost · 2025 Forecast ══╗",
        "  Target  : log₁p(DENGUE)",
        "  HPO     : Optuna TPE Sampler",
        f" Trials  : {OPTUNA_TRIALS}",
        f" CV      : {CV_FOLDS}-fold TimeSeriesSplit",
        "",
        "  ── Training Set ──",
        f"  Period  : 2008 → 2024",
        f"  n       : {n_train} obs.",
        f"  MAE     : {tr_m['MAE']:>12,.1f}",
        f"  RMSE    : {tr_m['RMSE']:>12,.1f}",
        f"  R²      : {tr_m['R2']:>12.4f}",
        f"  Pearson : {tr_m['Pearson_r']:>12.4f}",
        "",
        "  ── Test Set (2025 Jan–Oct) ──",
        f"  MAE     : {te_m['MAE']:>12,.1f}",
        f"  RMSE    : {te_m['RMSE']:>12,.1f}",
        f"  R²      : {te_m['R2']:>12.4f}",
        f"  MAPE    : {te_m['MAPE']:>11.2f}%",
        f"  SMAPE   : {te_m['SMAPE']:>11.2f}%",
        f"  Pearson : {te_m['Pearson_r']:>12.4f}",
        f"  Bias    : {te_m['Bias']:>12,.1f}",
        "",
        f"  Σ Obs   : {int(y_test.sum()):>12,}",
        f"  Σ Pred  : {int(y_pred.sum()):>12,}",
        f"  Cum Err : {abs(int(y_pred.sum())-int(y_test.sum())):>12,}",
        "",
        "  ── Best Hyperparameters ──",
        f"  n_est   : {best_params.get('n_estimators')}",
        f"  lr      : {best_params.get('learning_rate'):.4f}",
        f"  depth   : {best_params.get('max_depth')}",
        f"  min_cw  : {best_params.get('min_child_weight')}",
        f"  subsmp  : {best_params.get('subsample'):.2f}",
        f"  coltree : {best_params.get('colsample_bytree'):.2f}",
        f"  γ       : {best_params.get('gamma'):.3f}",
        f"  α (L1)  : {best_params.get('reg_alpha'):.4f}",
        f"  λ (L2)  : {best_params.get('reg_lambda'):.4f}",
        "╚══════════════════════════════╝",
    ]
    fig.text(0.808, 0.935, "\n".join(card),
             transform=fig.transFigure, ha="left", va="top",
             fontsize=7.6, color=P["WHITE"], fontfamily="monospace",
             bbox=dict(boxstyle="round,pad=0.6", facecolor=P["PANEL"],
                       edgecolor=P["BORDER"], alpha=0.98))

    fig.suptitle(
        "XGBoost Dengue Outbreak Forecasting — Bangladesh Prospective Evaluation 2025\n"
        r"Full Training: 2008–2024  $\rightarrow$  Prospective Test: Jan–Oct 2025  |  "
        r"log$_e$(1+y) Target  |  Optuna TPE HPO  |  SHAP Explainability",
        color=P["WHITE"], fontsize=12, fontweight="bold", y=0.990,
    )

    plt.savefig(OUTPUT_PNG, bbox_inches="tight", facecolor=P["BG"], dpi=150)
    print(f"Figure saved → {OUTPUT_PNG}")


# MAIN
def main():
    print("[1/9] Engineering features …")
    df = engineer_features(CSV_PATH)

    print("[2/9] Partitioning train (2008–2024) / test (2025 Jan–Oct) …")
    train = df.loc[:"2024-12-01"]
    test  = df.loc["2025-01-01":"2025-10-01"]

    X_tr = train[FEATURE_COLS]; y_tr_raw = train["DENGUE"].values
    y_tr_log = np.log1p(y_tr_raw)
    X_te = test[FEATURE_COLS];  y_te = test["DENGUE"].values

    print(f"      Train : {train.index.min().strftime('%Y-%b')} → "
          f"{train.index.max().strftime('%Y-%b')}  (n={len(train)})")
    print(f"      Test  : {test.index.min().strftime('%Y-%b')} → "
          f"{test.index.max().strftime('%Y-%b')}  (n={len(test)})")

    print(f"[3/9] Optuna TPE HPO ({OPTUNA_TRIALS} trials) …")
    best = optimise(X_tr, y_tr_log)
    print(f"      → n_est={best['n_estimators']}, lr={best['learning_rate']:.4f}, "
          f"depth={best['max_depth']}, min_cw={best['min_child_weight']}")

    print("[4/9] Training final model on full 2008–2024 set …")
    model = xgb.XGBRegressor(**best)
    model.fit(X_tr, y_tr_log)

    print("[5/9] Generating & back-transforming predictions …")
    y_fitted = np.expm1(np.clip(model.predict(X_tr), 0, None))
    y_pred   = np.expm1(np.clip(model.predict(X_te), 0, None))

    print("[6/9] Computing evaluation metrics …")
    tr_m = metrics(y_tr_raw, y_fitted)
    te_m = metrics(y_te, y_pred)

    print("[7/9] Computing SHAP explanations …")
    shap_vals = shap.TreeExplainer(model).shap_values(X_tr)

    print("[8/9] Printing report …")
    print_report(tr_m, te_m, best, test.index, y_pred, y_te, len(train))

    print("[9/9] Rendering figure …")
    build_figure(df, train, test, y_tr_raw, y_te,
                 y_fitted, y_pred, model,
                 tr_m, te_m, best, shap_vals)


if __name__ == "__main__":
    main()

[1/9] Engineering features …


[I 2026-06-22 21:30:09,019] A new study created in memory with name: no-name-ee4e7e9c-253f-4d54-9c40-16d7b1bd243c


[2/9] Partitioning train (2008–2024) / test (2025 Jan–Oct) …
      Train : 2009-Jan → 2024-Dec  (n=192)
      Test  : 2025-Jan → 2025-Oct  (n=10)
[3/9] Optuna TPE HPO (150 trials) …


[I 2026-06-22 21:30:11,882] Trial 0 finished with value: 3662.8516388021417 and parameters: {'n_estimators': 275, 'learning_rate': 0.12684995383408856, 'max_depth': 4, 'min_child_weight': 19, 'subsample': 0.5624074561769746, 'colsample_bytree': 0.47799726016810135, 'colsample_bylevel': 0.42904180608409975, 'gamma': 6.996321093312014, 'reg_alpha': 0.9644921218755025, 'reg_lambda': 2.1745470981373622}. Best is trial 0 with value: 3662.8516388021417.
[I 2026-06-22 21:30:15,900] Trial 1 finished with value: 3424.8837890818722 and parameters: {'n_estimators': 90, 'learning_rate': 0.13540804321249839, 'max_depth': 5, 'min_child_weight': 8, 'subsample': 0.5727299868828403, 'colsample_bytree': 0.49170225492671693, 'colsample_bylevel': 0.5521211214797689, 'gamma': 4.435673237241784, 'reg_alpha': 0.26660203678732375, 'reg_lambda': 0.0914863134229004}. Best is trial 1 with value: 3424.8837890818722.
[I 2026-06-22 21:30:17,762] Trial 2 finished with value: 3699.584937588871 and parameters: {'n_est

      → n_est=510, lr=0.0739, depth=2, min_cw=4
[4/9] Training final model on full 2008–2024 set …
[5/9] Generating & back-transforming predictions …
[6/9] Computing evaluation metrics …
[7/9] Computing SHAP explanations …
[8/9] Printing report …

  XGBoost Dengue Forecasting  —  2025 Jan–Oct Prospective Evaluation
  Target   : log₁p(DENGUE) → back-transformed with expm₁()
  Train    : 2008 → 2024  (n = 192)
  Test     : Jan 2025 → Oct 2025  (n = 10)

  Metric            In-Sample (Train)    Out-of-Sample (Test)
  ------------------------------------------------------------
  MAE                           851.1                12,330.4
  RMSE                        3,838.3                20,408.2
  R2                           0.8849                -10.0396
  MAPE                         33.81%                 109.94%
  SMAPE                        47.38%                  60.30%
  Pearson_r                    0.9742                  0.9280
  Bias                         -617.9          